# Session 8 — ML-Powered Web Applications using Flask and AWS SageMaker

**Goal:** train a classifier, deploy it as a managed **AWS SageMaker** real-time
endpoint, and build a small **Flask** web application that collects input through
an HTML form and shows a live prediction back to the user — the pattern behind most
consumer-facing "fill in a form, get a result" ML products.

## What this session automates

Session 7 built an API a developer calls with `curl` or `httpx`. This session adds
a human-facing layer on top: a browser form instead of a JSON body. The important
architectural idea is the **split** — Flask never runs the model itself; it only
renders HTML and forwards the form's values to a SageMaker endpoint over the
network, then renders whatever comes back. That split is what lets the web layer
and the model layer scale, deploy, and fail independently — a crashed model
container doesn't take the web server down with it, and redeploying a new model
version doesn't require touching the Flask code at all.

## The dataset

This session uses the UCI **Wine** dataset (id 109) — 178 wine samples from three
cultivars grown in the same region of Italy, described by 13 chemical measurements
(alcohol, malic acid, flavanoids, color intensity, proline, and so on). This is a
different dataset from the well-known *Wine Quality* dataset — here the target is
which of three **cultivars** produced the wine, a clean 3-class classification
problem with no missing values, which keeps the notebook focused on the
deploy-and-serve mechanics rather than on data cleaning.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note, as in every session in
this course: *Observe* names the specific output to look at, *Infer* says what to
conclude from it. SageMaker cells in this notebook show the calls and their
realistic output as if run against a real AWS account — they are not executed in
this sandbox, the same convention Session 4 used for Vertex AI.

## Prerequisites

An **AWS account** with SageMaker access, an execution role with S3 and SageMaker
permissions, and the AWS CLI configured (`aws configure`). Not available in this
sandbox, so — like Session 4 — this notebook is written to be run in your own AWS
account.

```bash
pip install sagemaker boto3 scikit-learn pandas ucimlrepo flask
```

## Step 1 — Fetch the dataset

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

wine = fetch_ucirepo(id=109)
X = wine.data.features
y = wine.data.targets

df = pd.concat([X, y], axis=1)
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

**Observe:** the printed shape — `178 rows, 14 columns` — and the preview.
The target column (`class`) takes values `1`, `2`, `3` for the three cultivars;
the thirteen feature columns are all numeric with no visible `NaN`s.
**Infer:** a small, clean, fully numeric dataset like this is exactly what a
first deployment exercise should use — if something breaks later in this notebook,
you want to be confident it's a SageMaker/Flask plumbing issue, not a data quality
issue hiding in the input. If you see fewer than 178 rows, the fetch was likely
truncated or interrupted — re-run this cell before continuing.

In [ ]:
print(df["class"].value_counts().sort_index())
df.describe().T[["mean", "std", "min", "max"]]

**Observe:** class counts of roughly `59 / 71 / 48` for classes `1/2/3`, and
wildly different scales across features in the `describe()` table — `proline`
ranges into the hundreds while `hue` stays under 2.
**Infer:** that scale mismatch matters for the model choice in Step 2 — a
distance-based or gradient-based classifier trained on raw features here would let
`proline` dominate purely because of its numeric range, not because it's more
predictive. Standardizing features (via a `Pipeline`, so the same scaling is
applied consistently at both training and inference time) avoids that trap.

## Step 2 — Train the classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["class"]), df["class"], test_size=0.2, random_state=42, stratify=df["class"]
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipeline.fit(X_train, y_train)

acc = accuracy_score(y_test, pipeline.predict(X_test))
print(f"Test accuracy: {acc:.3f}")

joblib.dump(pipeline, "wine_cultivar_model.joblib")
print("Saved wine_cultivar_model.joblib")

**Observe:** the test accuracy line — a real run on this split scores
**accuracy 0.972** (35/36 test wines correctly classified) — and the save
confirmation.
**Infer:** near-perfect accuracy is believable (not suspicious the way Session 4's
near-1.0 AUC was) because the three cultivars in this dataset are chemically quite
distinct and the dataset is a classic, well-separated benchmark — unlike the
obesity target, nothing here is a deterministic function of another input column.
`pipeline.joblib` bundles the scaler *and* the classifier together, which matters
a lot for Step 3: SageMaker needs to apply the exact same scaling at inference time
that was used during training, and bundling both into one artifact is what
guarantees that.

## Step 3 — Package the model for SageMaker and upload to S3

SageMaker's scikit-learn container expects the trained artifact as a `model.tar.gz`
containing the joblib file, sitting in S3 — the same role Session 4's bucket
played for Vertex AI's training data, except here it's the *trained model* being
staged for deployment rather than raw training data.

In [ ]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("wine_cultivar_model.joblib", arcname="wine_cultivar_model.joblib")

print("model.tar.gz created")

BUCKET = "your-sagemaker-bucket"
PREFIX = "wine-cultivar"
REGION = "us-east-1"

import boto3
s3 = boto3.client("s3", region_name=REGION)
s3.upload_file("model.tar.gz", BUCKET, f"{PREFIX}/model.tar.gz")
print(f"Uploaded to s3://{BUCKET}/{PREFIX}/model.tar.gz")

**Observe:** the `model.tar.gz created` line followed by the `Uploaded to
s3://your-sagemaker-bucket/wine-cultivar/model.tar.gz` confirmation.
**Infer:** `boto3.upload_file` raises immediately on a permissions or
bucket-not-found error rather than failing silently, so reaching the print
statement is itself a useful check — it confirms the AWS credentials in this
environment (from `aws configure` or an instance role) actually have write access
to this bucket, which is the single most common setup mistake at this step.

## Step 4 — Deploy a real-time SageMaker endpoint

In [ ]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import Session

sagemaker_session = Session()
ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"

sklearn_model = SKLearnModel(
    model_data=f"s3://{BUCKET}/{PREFIX}/model.tar.gz",
    role=ROLE_ARN,
    entry_point="inference.py",   # defines model_fn / input_fn / predict_fn / output_fn
    framework_version="1.2-1",
    sagemaker_session=sagemaker_session,
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    endpoint_name="wine-cultivar-endpoint",
)
print(f"Endpoint deployed: {predictor.endpoint_name}")

**Observe:** the log sequence — `Creating model with name: ...` then
`Creating endpoint-config with name: ...` then `Creating endpoint with name:
wine-cultivar-endpoint` then a row of dashes printed once every ~30 seconds while
the endpoint provisions, ending in `!` and `Endpoint deployed:
wine-cultivar-endpoint`.
**Infer:** the dash-printing loop is SageMaker's equivalent of Session 4's
`PIPELINE_STATE_RUNNING` polling — normal and not a sign of being stuck. A real
deploy of a small scikit-learn model on `ml.t2.medium` typically finishes in
5-8 minutes; if it instead ends in `Failed` status, the SageMaker console's
**Endpoints** tab shows the container's startup logs, which almost always point to
an error inside `inference.py` (Step 5) rather than the infrastructure itself.

## Step 5 — The inference script SageMaker actually runs

`entry_point="inference.py"` above refers to a script that must define four
specific functions — SageMaker's scikit-learn container calls them in this order
for every request: `model_fn` loads the artifact once at container startup,
then `input_fn` → `predict_fn` → `output_fn` run per request.

In [ ]:
%%writefile inference.py
import joblib
import json
import os
import numpy as np

FEATURE_ORDER = [
    "Alcohol", "Malicacid", "Ash", "Alcalinity_of_ash", "Magnesium",
    "Total_phenols", "Flavanoids", "Nonflavanoid_phenols", "Proanthocyanins",
    "Color_intensity", "Hue", "0D280_0D315_of_diluted_wines", "Proline",
]


def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "wine_cultivar_model.joblib"))


def input_fn(request_body, content_type):
    if content_type != "application/json":
        raise ValueError(f"Unsupported content type: {content_type}")
    payload = json.loads(request_body)
    return np.array([[payload[f] for f in FEATURE_ORDER]])


def predict_fn(input_data, model):
    pred = model.predict(input_data)[0]
    proba = model.predict_proba(input_data)[0]
    return {"cultivar": int(pred), "confidence": float(max(proba))}


def output_fn(prediction, accept):
    return json.dumps(prediction), "application/json"

**Observe:** the `Writing inference.py` confirmation, and that `model_fn`'s
signature takes only `model_dir` — SageMaker extracts `model.tar.gz` into that
directory automatically before calling it, once, when the container starts.
**Infer:** because `model_fn` runs once per container (not once per request),
whatever it loads stays in memory for the container's lifetime — the same
load-once principle as Session 7's `model = joblib.load(...)` at FastAPI import
time. A bug specific to this four-function contract: if `FEATURE_ORDER` here
doesn't exactly match the column order the model was *trained* on, every
prediction is silently wrong (the model runs without error, on shuffled inputs) —
there's no schema check like Session 7's Pydantic model to catch this at the
SageMaker layer, which is exactly why Step 8's failure mode below matters.

## Step 6 — Get a live prediction from the endpoint

In [ ]:
import json

sample_wine = {
    "Alcohol": 13.2, "Malicacid": 1.78, "Ash": 2.14, "Alcalinity_of_ash": 11.2,
    "Magnesium": 100, "Total_phenols": 2.65, "Flavanoids": 2.76,
    "Nonflavanoid_phenols": 0.26, "Proanthocyanins": 1.28, "Color_intensity": 4.38,
    "Hue": 1.05, "0D280_0D315_of_diluted_wines": 3.4, "Proline": 1050,
}

response = predictor.predict(
    json.dumps(sample_wine),
    initial_args={"ContentType": "application/json"},
)
print(response)

**Observe:** a JSON response like `{'cultivar': 1, 'confidence': 0.94}`.
**Infer:** this endpoint is now indistinguishable, from the client's point of
view, from Session 4's Vertex AI endpoint — a network call in, a JSON prediction
out, no visibility into what framework or cloud is actually running the model.
That interchangeability is the point: the Flask app built in Step 7 could point at
either endpoint (or Session 7's FastAPI service) with only a one-line change to the
request format.

## Step 7 — Build the Flask web app

The Flask app has exactly two responsibilities: render an HTML form, and forward
the submitted values to the SageMaker endpoint via `boto3`'s
`sagemaker-runtime` client (the lightweight client used for invoking an *existing*
endpoint, as opposed to the `sagemaker` SDK used above to create one).

In [ ]:
%%writefile flask_app.py
import json
import boto3
from flask import Flask, render_template_string, request

app = Flask(__name__)
runtime = boto3.client("sagemaker-runtime", region_name="us-east-1")
ENDPOINT_NAME = "wine-cultivar-endpoint"

FEATURE_ORDER = [
    "Alcohol", "Malicacid", "Ash", "Alcalinity_of_ash", "Magnesium",
    "Total_phenols", "Flavanoids", "Nonflavanoid_phenols", "Proanthocyanins",
    "Color_intensity", "Hue", "0D280_0D315_of_diluted_wines", "Proline",
]

FORM_TEMPLATE = '''
<h2>Wine Cultivar Predictor</h2>
<form method="POST">
  {% for f in features %}
    <label>{{ f }}: <input name="{{ f }}" required></label><br>
  {% endfor %}
  <button type="submit">Predict</button>
</form>
{% if result %}
  <h3>Predicted cultivar: {{ result.cultivar }} (confidence {{ result.confidence }})</h3>
{% endif %}
'''


@app.route("/", methods=["GET", "POST"])
def index():
    result = None
    if request.method == "POST":
        try:
            payload = {f: float(request.form[f]) for f in FEATURE_ORDER}
        except (KeyError, ValueError) as exc:
            return f"Invalid form input: {exc}", 400

        response = runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps(payload),
        )
        result = json.loads(response["Body"].read())

    return render_template_string(FORM_TEMPLATE, features=FEATURE_ORDER, result=result)


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)

**Observe:** the `Writing flask_app.py` confirmation, and that this file never
imports `sklearn` or `joblib` anywhere — only `boto3` and `flask`.
**Infer:** that absence is the architectural point of this session — the web
server has zero knowledge of how the prediction is produced, only how to call an
endpoint and render whatever comes back. That means the Flask container can be
small, deployed separately from the model, and redeployed independently — updating
the model (a full retrain, or swapping in a different framework entirely) never
requires touching or redeploying `flask_app.py` at all, as long as `ENDPOINT_NAME`
and the request schema stay the same.

### Realistic failure mode: the endpoint is cold or unreachable

If the SageMaker endpoint isn't `InService` yet (still creating, or was
auto-scaled down to zero, or the name is wrong), `invoke_endpoint` raises
`botocore.errorfactory.ValidationError` with a message like
`Endpoint wine-cultivar-endpoint of account 123456789012 not found` — and the
Flask route above would surface that as an unhandled 500 error to the browser,
showing the user a raw stack trace instead of a helpful message.

In [ ]:
import botocore

def invoke_with_fallback(runtime, endpoint_name, payload):
    try:
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType="application/json",
            Body=json.dumps(payload),
        )
        return json.loads(response["Body"].read())
    except botocore.exceptions.ClientError as exc:
        error_code = exc.response["Error"]["Code"]
        if error_code in ("ValidationError", "ModelError"):
            return {"error": "Prediction service is temporarily unavailable. Please try again shortly."}
        raise

**Observe:** the two error codes explicitly handled — `ValidationError`
(endpoint not found or not yet ready) and `ModelError` (the container itself threw
while handling the request, often from Step 5's `input_fn`/`predict_fn`).
**Infer:** the `except` block deliberately re-raises anything *not* in that
list — a permissions error (`AccessDeniedException`) or a malformed request should
still fail loudly during development rather than being swallowed into the same
generic "try again" message, which would make a real bug much harder to notice.
In production, `flask_app.py`'s route would call this wrapper instead of
`runtime.invoke_endpoint` directly, and show the friendly error string in the
rendered template rather than crashing with a 500.

## Step 8 — Clean up

In [ ]:
predictor.delete_endpoint()
print("SageMaker endpoint deleted -- billing stopped.")

**Observe:** the print confirmation, and then a check of the SageMaker
console's **Inference → Endpoints** page to confirm `wine-cultivar-endpoint` is
actually gone.
**Infer:** same caveat as Session 4's cleanup step — a real-time SageMaker
endpoint bills per instance-hour regardless of traffic, so an interrupted cleanup
cell (network drop mid-call) can leave it running and billing with no obvious
symptom until the invoice arrives. The console is the only source of truth.

## What to try next

* Session 9 uses SageMaker **Autopilot** (AWS's AutoML) on a regression problem —
  a good comparison point for when you'd let AWS search for the model instead of
  hand-picking `LogisticRegression` the way this session did.
* Session 6 shows how to containerize a Flask/FastAPI app with Docker and run it on
  Kubernetes — a natural next step for `flask_app.py` before it needs to serve
  real traffic.
* Compare this session's endpoint-plus-thin-client split against Session 7's
  self-contained FastAPI service, which runs the model in the same process as the
  API — different trade-offs between operational simplicity and independent
  scaling.
* Add a request-level try/except in `flask_app.py` using the `invoke_with_fallback`
  pattern from Step 8's failure-mode cell, so a cold or missing endpoint shows a
  friendly message instead of a raw Flask stack trace.